# Dataset

In [ ]:
! kaggle competitions download -c massive-problem-aicc-round-6
! 7z x massive-problem-aicc-round-6.zip

Mounted at /content/drive


# Task Description

### Storyline

You burst through the front doors of the AICC hospital. Checking your watch - *2:34 pm* - you breathe a sigh of relief. You're not late for your appraisal.  

Waiting in line to see the CEM, you see as engineers walk out aghast, faces pale as ghosts. Before long, a familiar, dreaded voice rings out. "Mr Dude, C., please proceed into the meeting room."

You walk in. A smartly dressed avatar, looking very much like a doctor, appears through a hologram. A virtual nameplate says: `Dr. GBT VII, Chief Executive Model`.  

`Good morning ☀️ You did an amazing job on your previous task — 79.5% accuracy is seriously impressive. Keep it up, you’re doing great.`

You glance at your wrist briefly. *No wonder we're still in the red.*  

`…However, I need to be honest with you — one of the patients from the missed cases has decided to sue us, and I’m in big trouble because of it.`  

*Welp, I've always wanted to get into farming. It's pretty a chill job.*  

`But there may be an opportunity coming up soon that could shift how all of this is being viewed.`

Your eyes widen.  

`It's something along the lines of figuring out a complete cure for this 'cancer' bug in human genes. Nothing urgent, just… whenever you get a moment.`  

*...what?*  

`I’ll consolidate the relevant materials and circle back by sending the data to your email once the qubits are aligned.` The avatar fizzles out.  

You stare blankly at where the avatar used to be. Perhaps this is why the other engineers looked so concerned.  

*Quite a massive problem...* you think to yourself. *But nothing a bit of AI can't fix, right?*

---

### Problem Statement

Train a classifier that can classify cells into one of 12 types, based on the expression level of **1434** genes.  

Note that the dataset is very large. You may run into issues with RAM usage if you do not manage your memory properly. Remember to `del` your unused variables!  

---

### Input Format

You are given 10 files:
- `RNA_seq_patient_0`-`RNA_seq_patient_8`: These 9 files were taken from **9 seperate patients**, and will be the training data for your model. Each row has the features `Gene1`-`Gene1434` (containing the expression level of each gene), `batch` (the patient the data was taken from) and `label` (the target cell type from 1-12).
- `test.csv`: This is the test set, taken from a separate 10th patient. It contains the features `Gene1`-`Gene1434` and `batch` only.

---

### Output Format

You must output as CSV file with two columns:
- `id`: The index of the predicted label, according to `test.csv` (equivalent to its index within `test_df`)
- `label`: The predicted label for the particular row of data.

---

### Scoring Method

The performance of your model will be determined by **macro-averaged F1 Score**.

The baseline solution in this notebook scores **0.2413**, rounded to 4 decimal places.

---

### Architecture Restrictions

There are no architectural restrictions, apart from the standard 'no LLM' rule. Good luck!  

# Baseline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

from xgboost import XGBClassifier

### Data Loading
We explicitly set the types when loading the CSV files, to use less memory.

In [ ]:
dtypes = {
    f"Gene{n}": "uint16" for n in range(1, 1435)
}
dtypes["batch"] = "int8"

In [ ]:
dfs = []
for p in range(0, 9):
    dfs.append(pd.read_csv(f"task_data/RNA_seq_patient_{p}.csv", dtype=dtypes))
df = pd.concat(dfs, ignore_index=True)
df.drop("Unnamed: 0", axis=1, inplace=True)

test_df = pd.read_csv("task_data/test.csv", dtype=dtypes)
test_df.drop("Unnamed: 0", axis=1, inplace=True)

Deleting unused variables, as shown below, can significantly reduce RAM usage.

In [ ]:
for d in dfs:
    del d
del dfs

We split the data into train and test via **random split**. Feel free to change this.

In [ ]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
del df

### Model Training

Let's train a model to predict on the train data we have. To make this faster, we will project the 1434 features to just 100 dimensions.

In [ ]:
X_train = train_df.drop("label", axis=1)
y_train = train_df["label"]
del train_df

In [ ]:
cls_pca = PCA(n_components=100)
X_train_pca = cls_pca.fit_transform(X_train.drop("batch", axis=1))
cls_pca.explained_variance_ratio_

array([5.07973570e-01, 2.19050890e-01, 7.82246461e-02, 7.62805640e-02,
       3.79588629e-02, 2.78276444e-02, 1.60887421e-02, 1.33219912e-02,
       1.98411581e-03, 9.95902379e-04, 8.63416353e-04, 8.11663007e-04,
       6.59516064e-04, 5.49482554e-04, 4.87104856e-04, 4.45136696e-04,
       4.11030672e-04, 3.94623504e-04, 3.73452093e-04, 3.02670116e-04,
       2.52547153e-04, 2.38858769e-04, 2.16784309e-04, 2.05581556e-04,
       1.98048053e-04, 1.86894565e-04, 1.73664476e-04, 1.71030787e-04,
       1.67034255e-04, 1.64340065e-04, 1.52500858e-04, 1.46887759e-04,
       1.40043778e-04, 1.36774000e-04, 1.31797153e-04, 1.28962087e-04,
       1.24040868e-04, 1.23004203e-04, 1.21745711e-04, 1.15496425e-04,
       1.14598834e-04, 1.11471496e-04, 1.07957505e-04, 1.04265720e-04,
       1.02695000e-04, 1.00433909e-04, 9.58970394e-05, 9.46753869e-05,
       9.27854118e-05, 9.12011597e-05, 8.99968819e-05, 8.68516509e-05,
       8.58270263e-05, 8.51708023e-05, 8.44059436e-05, 8.38903365e-05,
      

In [ ]:
X_train_final = np.concat((X_train_pca, X_train[["batch"]]), axis=1)
del X_train_pca, X_train

We will use XGBoost as a baseline.

In [ ]:
cls = XGBClassifier()
cls.fit(
    X_train_final,
    y_train - 1 # as XGBClassifier needs labels to start from 0
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
y_train_pred = cls.predict(X_train_final)

We evaluate the performance of the model on the data it was trained with.

In [ ]:
print("F1 Score:", f1_score(y_train_pred, y_train-1, average="macro"))

F1 Score: 0.80219705223146


# Evaluation

### Validation
Let's test our model on the validation dataset.

In [ ]:
X_val = val_df.drop("label", axis=1)
y_val = val_df["label"]
del val_df

X_val_final = np.concat((cls_pca.transform(X_val.drop("batch", axis=1)), X_val[["batch"]]), axis=1)
del X_val

y_pred = cls.predict(X_val_final) + 1
print("F1 Score:", f1_score(y_val, y_pred, average="macro"))

F1 Score: 0.5697062721741157


Looks decent, but... it's quite far from the score on the test set (0.2413). You can try submitting this on Kaggle to prove it to yourself.  

Why? That's for you to find out.

### Predict on Test

In [ ]:
X_test = test_df
X_test_final = np.concat((cls_pca.transform(X_test.drop("batch", axis=1)), X_test[["batch"]]), axis=1)
y_pred = cls.predict(X_test_final) + 1

### Save to CSV

In [ ]:
assert X_test.shape[0] == len(y_pred), "Mismatch in number of predictions"

submission_df = pd.DataFrame({
    'id': X_test.index.to_numpy(),
    'label': y_pred
})

submission_df.to_csv('submission.csv', index=False)

print("submission.csv created successfully.")
print(submission_df.head())

submission.csv created successfully.
   id  label
0   0      3
1   1      9
2   2      2
3   3      1
4   4      3
